# Training Neural ODEs on epidemiology models in Julia

## Introduction

In this example, we will generate data from a 5 compartment model: Susceptible -> Infected -> Recovered -> Hospitalized -> Deaths. Following this, we will construct a Neural ODE and train it on the data generated by the 5 compartment model. This example demonstrates the following things:

- How to write code which integrates neural networks with ODEs in Julia
- How to optimize the neural network parameters and train the resulting Neural ODE
- Why a Neural ODE, which knows nothing of the epidemic, fits the data but forecasts poorly

## Libraries

In [ ]:
Pollis.packages("Lux", "OrdinaryDiffEq", "SciMLSensitivity", "Optimization", "OptimizationOptimisers", "OptimizationOptimJL", "ComponentArrays", "Mooncake", "Plots")

In [ ]:
using Lux, OrdinaryDiffEq, SciMLSensitivity, Optimization, OptimizationOptimisers, OptimizationOptimJL, ComponentArrays, Mooncake, Random, Plots

## Data Generation
Generate data which simulates a Susceptible -> Infected -> Recovered -> Hospitalized -> Death scenario, as fractions of the population, observed every day for 25 days.

In [ ]:
rng = Xoshiro(1)

u0 = [0.99, 0.01, 0.0, 0.0, 0.0]

p0 = [
    0.85,   # tau_SI
    0.1,    # tau_IR
    0.025,  # tau_IH
    0.05,   # tau_ID
    0.02,   # tau_HR
    0.002,  # tau_HD
]

tspan = (0.0, 25.0)
t = 0.0:1.0:25.0

function SIRHD!(du, u, p, t)
    S, I, R, H, D = u
    tau_SI, tau_IR, tau_IH, tau_ID, tau_HR, tau_HD = p
    du[1] = -tau_SI * S * I                                # S
    du[2] = tau_SI * S * I - (tau_IR + tau_IH + tau_ID) * I  # I
    du[3] = tau_IR * I + tau_HR * H                        # R
    du[4] = tau_IH * I - (tau_HR + tau_HD) * H             # H
    du[5] = tau_ID * I + tau_HD * H                        # D
end

prob = ODEProblem(SIRHD!, u0, tspan, p0)
ode_data = Array(solve(prob, Tsit5(); saveat = t))

## Constructing a Neural ODE
The right-hand side of the ODE is a network, du/dt = NN(u), with a single hidden layer of 10 units and a tanh activation function.

In [ ]:
dudt = Chain(Dense(5 => 10, tanh), Dense(10 => 5))
p, st = Lux.setup(rng, dudt)

neural_rhs(u, p, t) = first(dudt(u, p, st))
prob_neuralode = ODEProblem(neural_rhs, u0, tspan)

predict_neuralode(p, ts = t) = Array(solve(prob_neuralode, Tsit5(); p = p, saveat = ts, tspan = (0.0, last(ts))))

loss_neuralode(p, _) = sum(abs2, ode_data .- predict_neuralode(p))

losses = Float64[]
callback = function (state, l)
    push!(losses, l)
    length(losses) % 50 == 0 && println("Loss after ", length(losses), " iterations: ", l)
    return false
end

## Training
Optimization.jl trains the network in two steps: Adam moves the parameters to a good region, then BFGS refines them.

In [ ]:
pinit = ComponentArray(p)
optf = OptimizationFunction(loss_neuralode, Optimization.AutoMooncake(; config = nothing))
optprob = OptimizationProblem(optf, pinit)

result_neuralode = solve(optprob, Adam(0.05); callback = callback, maxiters = 300)

optprob2 = remake(optprob; u0 = result_neuralode.u)
result_neuralode2 = solve(optprob2, Optim.BFGS(; initial_stepnorm = 0.01); callback = callback, maxiters = 200)
println("Final loss: ", losses[end])

plot(losses; yaxis = :log10, xlabel = "Iteration", ylabel = "Loss", label = "Training loss")

## Visualizing the predictions
The bars are the data, the lines the trained Neural ODE.

In [ ]:
data_pred = predict_neuralode(result_neuralode2.u)
plt = plot(; xlabel = "Day", ylabel = "Fraction of the population")
for (i, name) in enumerate(["S", "I", "R", "H", "D"])
    bar!(plt, t, ode_data[i, :]; alpha = 0.3, label = name * " data")
    plot!(plt, t, data_pred[i, :]; lw = 2, label = name * " prediction")
end
plt

## Forecasting
Run the trained Neural ODE for 50 days, twice the training period, and compare it with the true epidemic.

In [ ]:
t_long = 0.0:1.0:50.0
truth_long = Array(solve(remake(prob; tspan = (0.0, 50.0)), Tsit5(); saveat = t_long))
forecast = predict_neuralode(result_neuralode2.u, t_long)
plt = plot(; xlabel = "Day", ylabel = "Fraction of the population", legend = :right)
plot!(plt, t_long, truth_long[2, :]; color = :red, ls = :dash, label = "I (true)")
plot!(plt, t_long, forecast[2, :]; color = :red, lw = 2, label = "I (Neural ODE)")
plot!(plt, t_long, truth_long[3, :]; color = :blue, ls = :dash, label = "R (true)")
plot!(plt, t_long, forecast[3, :]; color = :blue, lw = 2, label = "R (Neural ODE)")
vline!(plt, [25]; color = :black, label = "End of the training data")

## Discussion
The fitted model gives a good fit to the training data, and the training iterations proceed fast. Beyond the training data, the forecast drifts away from the true epidemic: the network has learned the curve it was shown, not the mechanism of an epidemic. The next tutorial keeps the structure of the compartmental model and lets networks learn only its interaction terms.